In [ ]:
# KAGGLE CELL 1
import os

# 1. Install required dependencies
!pip install -q accelerate datasets soundfile librosa pandas wandb

# 2. Clone F5-TTS (Flow-Matching Architecture)
!git clone https://github.com/SWivid/F5-TTS.git
%cd F5-TTS
!pip install -e .

# 3. Download and extract your exact dataset
!gdown dgsdgdfgfgdf

import zipfile
os.makedirs('./dataset', exist_ok=True)
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('./dataset')

In [2]:
# KAGGLE/COLAB CELL 2
import os
import json
import librosa
import soundfile as sf
import pandas as pd
from datasets import Dataset, Audio
from tqdm.notebook import tqdm

print("🎧 Building F5-TTS hardcoded dataset structure...")

root_dir = "./dataset"
base_dir = "./data/mongolian_char"
wav_dir = os.path.join(base_dir, "wavs")

os.makedirs(wav_dir, exist_ok=True)

records =[]
vocab = set()
durations_dict = {} # ⬅️ THE MISSING DICTIONARY

# 1. Parse metadata & process audio
for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith("metadata.txt"):
            txt_path = os.path.join(dirpath, filename)
            with open(txt_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    parts = line.split("|")
                    if len(parts) >= 2:
                        wav_filename, text = parts[0], parts[1]
                        full_wav_path = os.path.join(dirpath, wav_filename)

                        if os.path.exists(full_wav_path):
                            audio, sr = librosa.load(full_wav_path, sr=24000, mono=True)
                            audio = librosa.util.normalize(audio)

                            duration = librosa.get_duration(y=audio, sr=24000)

                            out_wav_path = os.path.join(wav_dir, f"{len(records):05d}.wav")
                            sf.write(out_wav_path, audio, 24000, subtype='PCM_16')

                            records.append({
                                "audio": out_wav_path,
                                "text": text,
                                "duration": duration
                            })
                            # ⬅️ Map the path to duration for F5-TTS batching
                            durations_dict[out_wav_path] = duration
                            vocab.update(list(text))

# 2. Generate vocab.txt
vocab_list = sorted(list(vocab))
vocab_path = os.path.join(base_dir, "vocab.txt")
with open(vocab_path, "w", encoding="utf-8") as f:
    for c in vocab_list:
        f.write(c + "\n")

# 3. Generate duration.json (What just crashed your script)
duration_path = os.path.join(base_dir, "duration.json")
with open(duration_path, "w", encoding="utf-8") as f:
    json.dump(durations_dict, f, indent=4)
print(f"✅ Saved durations to {duration_path}")

# 4. Create Hugging Face Dataset format
print("📦 Building Arrow Dataset...")
df = pd.DataFrame(records)
hf_dataset = Dataset.from_pandas(df)
hf_dataset = hf_dataset.cast_column("audio", Audio(sampling_rate=24000))

raw_dir = os.path.join(base_dir, "raw")
hf_dataset.save_to_disk(raw_dir)

print(f"✅ Data fully saved to F5-TTS's required path: {base_dir}")

🎧 Building F5-TTS hardcoded dataset structure...
✅ Saved durations to ./data/mongolian_char/duration.json
📦 Building Arrow Dataset...


Saving the dataset (0/1 shards):   0%|          | 0/232 [00:00<?, ? examples/s]

✅ Data fully saved to F5-TTS's required path: ./data/mongolian_char


In [3]:
# KAGGLE/COLAB CELL 3
import yaml
import glob
import os

config_dir = "src/f5_tts/configs"
base_configs = glob.glob(f"{config_dir}/*Base.yaml")
base_config_path = base_configs[0]

with open(base_config_path, "r") as f:
    config = yaml.safe_load(f)

# --- INJECT DATASET CONFIG ---
config["datasets"]["name"] = "mongolian" # F5-TTS auto-maps this to data/mongolian_char/raw

# --- CONFIGURE CHARACTER TOKENIZER ---
if "model" not in config: config["model"] = {}
config["model"]["tokenizer"] = "char"

# --- TRAINING HYPERPARAMETERS ---
config["datasets"]["batch_size_per_gpu"] = 3840 # Safe for Colab 15GB T4 GPU
config["datasets"]["batch_size_type"] = "frame"
config["optim"]["epochs"] = 100
config["optim"]["learning_rate"] = 1e-5

custom_config_name = "F5TTS_Mongolian.yaml"
with open(os.path.join(config_dir, custom_config_name), "w") as f:
    yaml.dump(config, f)

print(f"✅ Custom config {custom_config_name} generated.")

✅ Custom config F5TTS_Mongolian.yaml generated.


In [4]:
import json

path = "./data/mongolian_char/duration.json"
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Wrap the dictionary in the exact key F5-TTS is looking for
if "duration" not in data:
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"duration": data}, f, indent=4)

In [5]:
import json
from datasets import Dataset

# Load the compiled Arrow dataset to guarantee exact row order
ds = Dataset.load_from_disk("./data/mongolian_char/raw")

# Convert the 'Column' object to a standard Python list of floats
duration_list =[float(x) for x in ds["duration"]]

# Extract the duration column as a pure list and overwrite the json F5-TTS reads
with open("./data/mongolian_char/duration.json", "w", encoding="utf-8") as f:
    json.dump({"duration": duration_list}, f)

print(f"✅ Fixed! duration.json now contains a list of {len(duration_list)} floats.")

✅ Fixed! duration.json now contains a list of 232 floats.


In [6]:
import os
from datasets import Dataset, Audio

base_dir = "./data/mongolian_char"
raw_dir = os.path.join(base_dir, "raw")

print("🛠️ Fixing dataset structure for F5-TTS...")

# Load the existing dataset
ds = Dataset.load_from_disk(raw_dir)

# Disable audio decoding so we don't load all audio arrays into RAM
ds = ds.cast_column("audio", Audio(decode=False))

# Extract pure strings for the paths instead of Audio objects
audio_paths = [row["path"] for row in ds["audio"]]
texts = ds["text"]
durations = [float(d) for d in ds["duration"]]

# Create the corrected dataset
fixed_ds = Dataset.from_dict({
    "audio_path": audio_paths,  # ⬅️ F5-TTS needs this exact key
    "text": texts,
    "duration": durations
})

# Overwrite the broken one
fixed_ds.save_to_disk(raw_dir)

print(f"✅ Fixed! Arrow dataset now has 'audio_path' (string), 'text', and 'duration'.")

🛠️ Fixing dataset structure for F5-TTS...


Saving the dataset (0/1 shards):   0%|          | 0/232 [00:00<?, ? examples/s]

✅ Fixed! Arrow dataset now has 'audio_path' (string), 'text', and 'duration'.


In [7]:
import os
from datasets import Dataset

# Define absolute paths
base_dir = "/content/F5-TTS/data/mongolian_char"
raw_dir = os.path.join(base_dir, "raw")
wav_dir = os.path.join(base_dir, "wavs")

print("🛠️ Fixing dataset to use absolute paths...")

# Load the dataset
ds = Dataset.load_from_disk(raw_dir)

# Rebuild the correct absolute paths by combining wav_dir and the filename
correct_audio_paths =[
    os.path.join(wav_dir, os.path.basename(path)) for path in ds["audio_path"]
]

# Update the dataset
fixed_ds = Dataset.from_dict({
    "audio_path": correct_audio_paths,
    "text": ds["text"],
    "duration": ds["duration"]
})

# Overwrite
fixed_ds.save_to_disk(raw_dir)

print(f"✅ Fixed! The paths now look exactly like this:")
print(f"👉 {correct_audio_paths[0]}")

🛠️ Fixing dataset to use absolute paths...


Saving the dataset (0/1 shards):   0%|          | 0/232 [00:00<?, ? examples/s]

✅ Fixed! The paths now look exactly like this:
👉 /content/F5-TTS/data/mongolian_char/wavs/00000.wav


In [ ]:
# KAGGLE/COLAB CELL 4
!WANDB_DISABLED=true accelerate launch \
    --num_processes=1 \
    --num_machines=1 \
    --mixed_precision=fp16 \
    --dynamo_backend=no \
    src/f5_tts/train/train.py \
    --config-name F5TTS_Mongolian.yaml


Epoch 4/100:  54% 36/67 [00:27<00:21,  1.42update/s, loss=4.64, update=236]
Epoch 4/100:  54% 36/67 [00:27<00:21,  1.42update/s, loss=5.16, update=237]
Epoch 4/100:  55% 37/67 [00:28<00:20,  1.46update/s, loss=5.16, update=237]
Epoch 4/100:  55% 37/67 [00:28<00:20,  1.46update/s, loss=4.86, update=238]
Epoch 4/100:  57% 38/67 [00:28<00:20,  1.44update/s, loss=4.86, update=238]
Epoch 4/100:  57% 38/67 [00:28<00:20,  1.44update/s, loss=4.81, update=239]
Epoch 4/100:  58% 39/67 [00:29<00:19,  1.45update/s, loss=4.81, update=239]
Epoch 4/100:  58% 39/67 [00:29<00:19,  1.45update/s, loss=4.76, update=240]
Epoch 4/100:  60% 40/67 [00:30<00:18,  1.43update/s, loss=4.76, update=240]
Epoch 4/100:  60% 40/67 [00:30<00:18,  1.43update/s, loss=4.88, update=241]
Epoch 4/100:  61% 41/67 [00:30<00:18,  1.43update/s, loss=4.88, update=241]
Epoch 4/100:  61% 41/67 [00:30<00:18,  1.43update/s, loss=4.88, update=242]
Epoch 4/100:  63% 42/67 [00:31<00:17,  1.44update/s, loss=4.88, update=242]
Epoch 4/100

In [ ]:
# KAGGLE CELL 5
import os

# Auto-locate the latest trained checkpoint
checkpoint_dir = "./ckpts/mongolian_custom/"
checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith(".pt")])
latest_ckpt = os.path.join(checkpoint_dir, checkpoints[-1])

print(f"🔊 Running inference on: {latest_ckpt}")

# Use one of your clean wav files as the voice reference (Prompt)
ref_audio = "./data/mongolian_custom/00000.wav"
ref_text = "Үүнийг өөрийн аудионд тохируулан солино уу." # Replace with actual text of the ref_audio
gen_text = "Өнөөдөр гадаа хасах хорин хэм хүйтэн байна."

# Execute standard inference
!python src/f5_tts/infer/infer_cli.py \
    --model "F5-TTS" \
    --ckpt_file "{latest_ckpt}" \
    --vocab_file "./data/mongolian_custom/vocab.txt" \
    --ref_audio "{ref_audio}" \
    --ref_text "{ref_text}" \
    --gen_text "{gen_text}" \
    --output_dir "output"

import IPython.display as ipd
ipd.display(ipd.Audio("output/out.wav", rate=24000))

In [ ]:
!ls -R /content/